<a href="https://colab.research.google.com/github/juliandavidsilvaguzman-star/Transformaci-n-de-textos-en-embeddings/blob/main/Transformar_textos_en_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformación de textos en embeddings
# Procesamiento de Lenguaje Natural
# Docente: Paul Alexander Diaz Montaña
# Alumno: Julián Davied Silva Guzman
# 06/09/2026



Transformación de textos en embeddings es el proceso de convertir texto (palabras, frases o documentos) en vectores numéricos que una computadora puede procesar y comparar. Los embeddings representan el significado semántico del texto, de modo que textos con significados similares quedan ubicados cerca unos de otros en un espacio matemático.

Funciones que se ejecutan:

- Convertir textos en embeddings con un modelo multilingüe.
- Calcular similitud semántica mediante similitud coseno.
- Buscar los textos más relacionados con una consulta.
- Cargar textos desde un archivo CSV.
- Exportar y descargar los embeddings.


## 1. Instalar las librerías


In [1]:
!pip install -q -U sentence-transformers pandas scikit-learn # Instala las librerías necesarias: sentence-transformers para los embeddings, pandas para el manejo de datos y scikit-learn para utilidades como la similitud del coseno.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


### Análisis de la instalación de librerías

Este comando instala o actualiza silenciosamente las librerías `sentence-transformers`, `pandas` y `scikit-learn`. Aunque la salida muestra un error de `pip` relacionado con la resolución de dependencias de `pandas` (versión 3.0.5 incompatible con `google-colab` y `cudf-cu12`), las librerías principales (`sentence-transformers`) suelen funcionar correctamente para los propósitos de este cuaderno. El error indica una incompatibilidad de versiones entre `pandas` y otras librerías preinstaladas en Colab.

## 2. Importar las librerías


In [2]:
import numpy as np # Importa la librería NumPy para operaciones numéricas.
import pandas as pd # Importa la librería Pandas para manipulación y análisis de datos en DataFrames.
import torch # Importa la librería PyTorch, utilizada para operaciones con tensores y detección de GPU.

from sentence_transformers import SentenceTransformer # Importa la clase SentenceTransformer para cargar modelos de embeddings.
from sklearn.metrics.pairwise import cosine_similarity # Importa la función cosine_similarity de scikit-learn para calcular la similitud del coseno.

### Análisis de la importación de librerías

Se importan las librerías esenciales para el notebook:
*   `numpy` (np): Para operaciones numéricas, especialmente con arrays de los embeddings.
*   `pandas` (pd): Para la manipulación y análisis de datos en DataFrames.
*   `torch`: Parte del ecosistema PyTorch, fundamental para el funcionamiento de `sentence-transformers` y para la detección de GPU.
*   `SentenceTransformer`: La clase clave de la librería `sentence-transformers` para cargar y utilizar modelos de embeddings.
*   `cosine_similarity`: Una función de `scikit-learn` para calcular la similitud del coseno entre vectores, que es crucial para determinar la similitud semántica.

## 3. Seleccionar CPU o GPU


In [3]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu" # Detecta si hay una GPU (CUDA) disponible; de lo contrario, usa la CPU.
print(f"Dispositivo utilizado: {dispositivo}") # Imprime el dispositivo (CPU o GPU) que se utilizará para el procesamiento.

Dispositivo utilizado: cuda


### Análisis de la selección de dispositivo

El código detecta si hay una GPU (CUDA) disponible en el entorno de Google Colab. Si `torch.cuda.is_available()` es verdadero, se selecciona `cuda`; de lo contrario, se selecciona `cpu`. La salida `Dispositivo utilizado: cuda` confirma que se ha detectado una GPU. Esto es beneficioso porque las operaciones de procesamiento de embeddings se realizarán en la GPU, lo que resulta en una ejecución significativamente más rápida y eficiente.

## 4. Cargar un modelo multilingüe


In [4]:
nombre_modelo = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2" # Define el nombre del modelo de SentenceTransformer multilingüe a cargar.
modelo = SentenceTransformer(nombre_modelo, device=dispositivo) # Carga el modelo de SentenceTransformer, especificando el dispositivo (CPU/GPU).
print("Modelo cargado correctamente.") # Confirma que el modelo ha sido cargado.

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado correctamente.


### Análisis de la carga del modelo multilingüe

Se carga el modelo pre-entrenado `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Este modelo está diseñado para generar embeddings para textos en múltiples idiomas, lo que lo hace muy versátil. La salida `Modelo cargado correctamente.` confirma que el modelo se inicializó con éxito y se asignó al dispositivo (`cuda` en este caso) detectado previamente. La advertencia sobre `HF_TOKEN` es un aviso estándar de Hugging Face y no afecta la funcionalidad del modelo para este propósito.

## 5. Definir los textos


In [5]:
textos = [ # Define una lista de textos de ejemplo.
    "La inteligencia artificial permite analizar imágenes de cámaras de seguridad.",
    "Los modelos de visión artificial pueden detectar objetos en vídeos.",
    "La transformación digital requiere el desarrollo de competencias digitales.",
    "El trabajo remoto modifica las dinámicas de las organizaciones públicas."
]

print(f"Cantidad de textos: {len(textos)}") # Imprime la cantidad de textos en la lista.

Cantidad de textos: 4


### Análisis de la definición de textos

Se define una lista de `textos` de ejemplo que se utilizarán para demostrar la funcionalidad de los embeddings. La salida `Cantidad de textos: 4` confirma que se han cargado cuatro frases, las cuales servirán como base para generar sus respectivas representaciones vectoriales en los pasos siguientes.

## 6. Transformar los textos en embeddings


In [6]:
embeddings = modelo.encode( # Genera los embeddings (vectores numéricos) para la lista de textos.
    textos, # Los textos a codificar.
    batch_size=32, # El número de textos procesados en cada lote para optimizar la memoria.
    show_progress_bar=True, # Muestra una barra de progreso durante la codificación.
    convert_to_numpy=True, # Convierte los embeddings resultantes a un array de NumPy.
    normalize_embeddings=True # Normaliza los embeddings a una longitud unitaria, lo cual es útil para la similitud del coseno.
)

print("Cantidad de textos:", len(textos)) # Imprime la cantidad de textos procesados.
print("Dimensión de cada embedding:", embeddings.shape[1]) # Imprime la dimensión de cada vector embedding.
print("Forma de la matriz:", embeddings.shape) # Imprime la forma (número de textos, dimensión del embedding) de la matriz de embeddings.

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cantidad de textos: 4
Dimensión de cada embedding: 384
Forma de la matriz: (4, 384)


### Análisis de la transformación de textos en embeddings

Esta celda realiza la conversión de la lista de `textos` a sus representaciones numéricas, conocidas como embeddings. Los embeddings son vectores de números de punto flotante que capturan el significado semántico de cada frase.

*   **`Cantidad de textos: 4`**: Confirma que se procesaron los 4 textos de la lista.
*   **`Dimensión de cada embedding: 384`**: Indica que cada frase se transformó en un vector de 384 números. Una dimensión mayor generalmente permite una representación más rica del significado.
*   **`Forma de la matriz: (4, 384)`**: Muestra que el resultado es una matriz de 4 filas (una por cada texto) y 384 columnas (una por cada dimensión del embedding). La normalización a longitud unitaria facilita las comparaciones de similitud posterior.

## 7. Visualizar los primeros valores


In [7]:
for indice, texto in enumerate(textos): # Itera sobre cada texto y su índice en la lista 'textos'.
    print(f"\nTexto {indice + 1}: {texto}") # Imprime el texto actual con un número de índice.
    print("Primeros 10 valores del embedding:") # Indica que se mostrarán los primeros valores del embedding.
    print(embeddings[indice][:10]) # Imprime los primeros 10 valores del vector embedding correspondiente al texto actual.


Texto 1: La inteligencia artificial permite analizar imágenes de cámaras de seguridad.
Primeros 10 valores del embedding:
[-0.07699889 -0.01374324 -0.07512615 -0.06521741  0.06013423  0.05085905
  0.06610879 -0.03552571  0.00828497  0.06997788]

Texto 2: Los modelos de visión artificial pueden detectar objetos en vídeos.
Primeros 10 valores del embedding:
[-0.0876506  -0.0713698   0.04982905 -0.02533723  0.0565312   0.00728244
  0.03333691 -0.08954591 -0.00698665  0.06704062]

Texto 3: La transformación digital requiere el desarrollo de competencias digitales.
Primeros 10 valores del embedding:
[-0.01056632  0.04289617 -0.00268902 -0.06622872 -0.03491314 -0.01553994
 -0.04560295 -0.00880236 -0.03455179  0.1145236 ]

Texto 4: El trabajo remoto modifica las dinámicas de las organizaciones públicas.
Primeros 10 valores del embedding:
[-0.03722846 -0.04599549  0.01082654 -0.00406489  0.03730101 -0.02209884
 -0.08569922  0.00262171 -0.00168078  0.05335984]


### Análisis de la visualización de los primeros valores

Esta celda itera a través de cada texto y su embedding correspondiente, mostrando el texto original y los primeros 10 valores de su vector de embedding. Los números que se muestran son los valores reales dentro del vector numérico que representa cada frase. Por sí solos, no son directamente interpretables por humanos, pero demuestran cómo cada texto es codificado en una secuencia única de números de punto flotante, capturando su información semántica.

## 8. Calcular la similitud semántica


In [8]:
matriz_similitud = cosine_similarity(embeddings) # Calcula la matriz de similitud del coseno entre todos los pares de embeddings.

resultado_similitud = pd.DataFrame( # Crea un DataFrame de Pandas a partir de la matriz de similitud.
    matriz_similitud, # Los datos de la matriz de similitud.
    index=[f"Texto {i + 1}" for i in range(len(textos))], # Asigna etiquetas a las filas (Texto 1, Texto 2, etc.).
    columns=[f"Texto {i + 1}" for i in range(len(textos))] # Asigna etiquetas a las columnas (Texto 1, Texto 2, etc.).
)

resultado_similitud.round(4) # Muestra el DataFrame de similitud del coseno redondeado a 4 decimales.

,Texto 1,Texto 2,Texto 3,Texto 4
Texto 1,1.0000,0.5998,0.3252,0.0814
Texto 2,0.5998,1.0000,0.2062,0.0733
Texto 3,0.3252,0.2062,1.0000,0.3669
Texto 4,0.0814,0.0733,0.3669,1.0000


### Análisis del cálculo de la similitud semántica

Esta celda calcula la similitud del coseno entre todos los pares de embeddings generados. La similitud del coseno es una medida de qué tan similares son la dirección de dos vectores, con un valor de 1 indicando máxima similitud y -1 máxima disimilitud.

*   **Diagonal (1.0000):** Un texto siempre es idéntico a sí mismo.
*   **Valores altos (ej. Texto 1 vs Texto 2: 0.5998):** Indican una fuerte relación semántica. Por ejemplo, "La inteligencia artificial permite analizar imágenes de cámaras de seguridad" y "Los modelos de visión artificial pueden detectar objetos en vídeos" son conceptos relacionados.
*   **Valores bajos (ej. Texto 1 vs Texto 4: 0.0814):** Sugieren poca o ninguna relación semántica entre los textos. Por ejemplo, "La inteligencia artificial..." y "El trabajo remoto..." no comparten un significado cercano.

## 9. Buscar los textos más parecidos a una consulta


In [9]:
consulta = "Uso de inteligencia artificial para detectar objetos en cámaras" # Define el texto de la consulta.

embedding_consulta = modelo.encode( # Genera el embedding para la consulta.
    [consulta],
    convert_to_numpy=True,
    normalize_embeddings=True
) # Normaliza el embedding de la consulta.

similitudes = cosine_similarity(embedding_consulta, embeddings)[0] # Calcula la similitud del coseno entre la consulta y todos los textos existentes.

resultados = pd.DataFrame({ # Crea un DataFrame para mostrar los resultados de similitud.
    "texto": textos, # La columna 'texto' contiene los textos originales.
    "similitud": similitudes # La columna 'similitud' contiene los valores de similitud calculados.
}).sort_values( # Ordena el DataFrame por la columna 'similitud'.
    by="similitud", # Columna para ordenar.
    ascending=False # Ordena de mayor a menor similitud.
).reset_index(drop=True) # Reinicia el índice del DataFrame y elimina el índice antiguo.

print("Consulta:", consulta) # Imprime la consulta original.
resultados # Muestra el DataFrame con los textos ordenados por similitud a la consulta.

Consulta: Uso de inteligencia artificial para detectar objetos en cámaras


,texto,similitud
0,La inteligencia artificial permite analizar im...,0.867561
1,Los modelos de visión artificial pueden detect...,0.713125
2,La transformación digital requiere el desarrol...,0.273898
3,El trabajo remoto modifica las dinámicas de la...,0.065144


### Análisis de la búsqueda de textos parecidos a una consulta

Se define una consulta ("Uso de inteligencia artificial para detectar objetos en cámaras") y se genera su embedding. Luego, se calcula la similitud del coseno entre el embedding de la consulta y los embeddings de los textos de ejemplo. La tabla de resultados muestra los textos ordenados de mayor a menor similitud con la consulta. Se observa claramente que los textos 0 y 1, que tratan sobre inteligencia artificial, cámaras y detección de objetos, tienen las similitudes más altas (0.867561 y 0.713125 respectivamente). Esto demuestra la efectividad del modelo para identificar y clasificar textos por su relevancia semántica respecto a una consulta dada.

## 10. Guardar y descargar los embeddings


In [10]:
columnas_embedding = [f"embedding_{i}" for i in range(embeddings.shape[1])] # Crea una lista de nombres para las columnas de los embeddings (ej. 'embedding_0', 'embedding_1').
df_embeddings = pd.DataFrame(embeddings, columns=columnas_embedding) # Crea un DataFrame de Pandas con los embeddings y los nombres de columna.
df_embeddings.insert(0, "texto", textos) # Inserta la columna de textos originales al inicio del DataFrame de embeddings.

nombre_archivo_salida = "textos_embeddings.csv" # Define el nombre para el archivo CSV de salida.
df_embeddings.to_csv(nombre_archivo_salida, index=False, encoding="utf-8-sig") # Guarda el DataFrame en un archivo CSV, sin el índice y con codificación UTF-8.
print(f"Archivo creado: {nombre_archivo_salida}") # Imprime un mensaje confirmando la creación del archivo.

from google.colab import files # Importa el módulo 'files' de Google Colab para la descarga.
files.download(nombre_archivo_salida) # Inicia la descarga del archivo CSV a la máquina local del usuario.

Archivo creado: textos_embeddings.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Análisis de guardar y descargar los embeddings

Esta celda tiene como objetivo almacenar los textos originales junto con sus embeddings correspondientes. Se crea un nuevo DataFrame (`df_embeddings`) que incluye la columna de textos y 384 columnas adicionales para los valores de los embeddings (nombradas `embedding_0` a `embedding_383`). Luego, este DataFrame se guarda en un archivo CSV llamado `textos_embeddings.csv` y se descarga automáticamente. Esta funcionalidad es crucial para persistir los embeddings, permitiendo su reutilización en futuros análisis o modelos sin tener que recalcularlos, lo que ahorra tiempo y recursos computacionales.

## 11. Opción alternativa: cargar textos desde un CSV

El archivo debe contener una columna llamada **texto**. Ejecuta esta sección si deseas trabajar con un CSV propio. Esta celda sustituye la lista de textos definida anteriormente y genera `textos_con_embeddings.csv`.


In [11]:
from google.colab import files # Importa el módulo 'files' de Google Colab para cargar archivos.

archivo_cargado = files.upload() # Abre una ventana de diálogo para que el usuario suba un archivo.
nombre_archivo_entrada = next(iter(archivo_cargado)) # Obtiene el nombre del archivo cargado.

df = pd.read_csv(nombre_archivo_entrada) # Lee el archivo CSV cargado y lo convierte en un DataFrame de pandas.

# Verifica si la columna 'texto' existe en el DataFrame.
if "texto" not in df.columns:
    raise ValueError("El archivo CSV debe contener una columna llamada 'texto'.") # Lanza un error si la columna no existe.

df["texto"] = df["texto"].fillna("").astype(str) # Rellena los valores nulos en la columna 'texto' con cadenas vacías y asegura que todos los valores sean de tipo cadena.
textos_csv = df["texto"].tolist() # Convierte la columna 'texto' del DataFrame en una lista de cadenas.

embeddings_csv = modelo.encode( # Genera embeddings para los textos de la lista 'textos_csv'.
    textos_csv, # Lista de textos a codificar.
    batch_size=32, # Número de textos procesados en cada lote.
    show_progress_bar=True, # Muestra una barra de progreso durante la codificación.
    convert_to_numpy=True, # Convierte los embeddings a un array de NumPy.
    normalize_embeddings=True # Normaliza los embeddings a una longitud unitaria.
)

# Crea una lista de nombres de columnas para los embeddings, como 'embedding_0', 'embedding_1', etc.
columnas_embedding_csv = [
    f"embedding_{i}" for i in range(embeddings_csv.shape[1])
]

df_vectores = pd.DataFrame( # Crea un DataFrame con los embeddings generados.
    embeddings_csv,
    columns=columnas_embedding_csv # Asigna los nombres de columna definidos anteriormente.
)

resultado_csv = pd.concat( # Concatena el DataFrame original con el DataFrame de embeddings.
    [df.reset_index(drop=True), df_vectores],
    axis=1 # Concatena a lo largo del eje de columnas.
)

nombre_resultado = "textos_con_embeddings.csv" # Define el nombre del archivo CSV de salida.
resultado_csv.to_csv(nombre_resultado, index=False, encoding="utf-8-sig") # Guarda el DataFrame resultante en un archivo CSV.
print(f"Archivo creado: {nombre_resultado}") # Imprime un mensaje indicando que el archivo ha sido creado.
files.download(nombre_resultado) # Descarga el archivo CSV a la máquina local del usuario.

Saving textos.csv to textos.csv


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Archivo creado: textos_con_embeddings.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Análisis de la opción alternativa: cargar textos desde un CSV

Esta celda proporciona una forma interactiva de procesar datos de texto desde un archivo CSV proporcionado por el usuario. Cuando se ejecuta, se abre un cuadro de diálogo para que el usuario suba un archivo. El código:

1.  **Carga el archivo CSV:** Lee el contenido del CSV en un DataFrame de `pandas`.
2.  **Verifica la columna 'texto':** Asegura que el CSV contiene una columna llamada 'texto', que es donde se espera que estén los datos textuales a procesar. Si no existe, genera un error.
3.  **Genera Embeddings:** Utiliza el modelo cargado (`modelo.encode`) para crear embeddings para todos los textos en la columna 'texto' del CSV, de manera similar a cómo se hizo con la lista de textos de ejemplo.
4.  **Combina Datos:** Concatena el DataFrame original (con los textos del CSV) con el nuevo DataFrame que contiene los embeddings generados.
5.  **Guarda y Descarga:** Guarda el DataFrame resultante (textos + embeddings) en un nuevo archivo CSV (`textos_con_embeddings.csv`) y lo descarga al equipo del usuario. Esto permite a los usuarios aplicar todo el flujo de trabajo de embeddings a sus propios conjuntos de datos de manera sencilla.

## Notas

- Los embeddings normalizados facilitan la comparación mediante similitud coseno.
- Un valor de similitud más cercano a `1` indica mayor semejanza semántica


## Análisis General y Conclusiones

Este ejemplo práctico ha demostrado un flujo de trabajo completo para la **transformación de texto en embeddings y su posterior aplicación en la búsqueda de similitud semántica**, utilizando el poder de los modelos de `SentenceTransformer`.

**Análisis General:**

1.  **Preparación del Entorno:** El proceso comienza con la instalación de las librerías necesarias (`sentence-transformers`, `pandas`, `scikit-learn`) y la importación de los módulos clave. La detección automática del dispositivo (`cuda` o `cpu`) asegura la optimización del rendimiento, aprovechando la GPU cuando está disponible.
2.  **Carga y Uso del Modelo:** Se cargó un modelo multilingüe (`paraphrase-multilingual-MiniLM-L12-v2`), lo que permite el procesamiento de texto en varios idiomas. Esto es fundamental para la flexibilidad de la solución.
3.  **Generación de Embeddings:** El corazón del ejemplo práctico es la conversión de textos en vectores numéricos de 384 dimensiones. Estos embeddings son representaciones densas que capturan el significado semántico del texto, una pieza clave para que las máquinas puedan 'entender' el lenguaje humano.
4.  **Similitud Semántica:** La similitud del coseno fue utilizada para cuantificar la relación semántica entre los textos. Se observó claramente cómo frases con significados similares resultan en valores de similitud altos, mientras que frases dispares tienen valores bajos. Esto fue evidenciado tanto en la matriz de similitud como en la búsqueda por consulta.
5.  **Búsqueda y Relevancia:** La demostración de búsqueda por consulta mostró la capacidad del sistema para encontrar los textos más relevantes a una pregunta dada, ordenándolos por su similitud semántica. Esto es aplicable en motores de búsqueda, sistemas de recomendación, y análisis de documentos.
6.  **Persistencia y Escalabilidad:** El ejemplo práctico ofrece soluciones para guardar los embeddings generados en un archivo CSV, lo que permite su reutilización sin necesidad de recálculo. Además, la funcionalidad de cargar textos desde un CSV externo hace que el proceso sea escalable para trabajar con grandes volúmenes de datos propios del usuario.

**Conclusiones:**

*   **Potencial de los Embeddings:** Los embeddings de texto son una herramienta extremadamente poderosa en el Procesamiento del Lenguaje Natural (PLN), permitiendo a las máquinas trabajar con el significado contextual de las palabras y frases, no solo con su forma literal.
*   **Aplicaciones Versátiles:** Las técnicas demostradas son la base de muchas aplicaciones de IA, incluyendo:
    *   **Búsqueda Semántica:** Mejora la relevancia de los resultados de búsqueda.
    *   **Recomendación de Contenido:** Sugiere elementos basados en el significado, no solo en palabras clave.
    *   **Clasificación de Texto:** Agrupa documentos similares.
    *   **Detección de Plagio:** Identifica contenido con significado similar.
*   **Facilidad de Uso:** Librerías como `sentence-transformers` simplifican la implementación de estas técnicas avanzadas, democratizando el acceso a la IA del lenguaje.
*   **Impacto Práctico:** La capacidad de procesar y comparar textos a nivel semántico tiene un impacto significativo en la automatización y la mejora de la eficiencia en tareas relacionadas con la información y el conocimiento.

En resumen, este ejemplo práctico ha proporcionado una comprensión práctica y funcional de cómo transformar texto en representaciones numéricas significativas y utilizarlas para medir la similitud semántica, un concepto fundamental en el PLN moderno.